In [3]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import astropy.io.fits as fits
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
import pickle
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import ListedColormap
from matplotlib.ticker import AutoMinorLocator, FormatStrFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

datadir  = '/home/mnedal/data/'
filename = 'OVSA/20260118.fits'

In [4]:
data = fits.open(f'{datadir}/{filename}')
data.info()

Filename: /home/mnedal/data//OVSA/20260118.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      26   (53385, 731, 1, 2)   float64   
  1  SFREQ         1 BinTableHDU     11   731R x 1C   [E]   
  2  UT            1 BinTableHDU     13   53385R x 2C   [J, J]   
  3  FREQ_CAL      1 BinTableHDU     15   3072R x 3C   [E, E, E]   


In [3]:
data[0].header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    4 / number of array dimensions                     
NAXIS1  =                53385                                                  
NAXIS2  =                  731                                                  
NAXIS3  =                    1                                                  
NAXIS4  =                    2                                                  
EXTEND  =                    T                                                  
FILENAME= '20260118.fits'                                                       
ORIGIN  = 'NJIT    '           / Location where file was made                   
DATE    = '2026-01-19T12:06:06.605' / Date when file was made                   
OBSERVER= 'EOVSA Team'         / Who to appreciate/blame                        
TELESCOP= 'EOVSA   '        

In [4]:
st = data[0].header['DATE_OBS']
et = data[0].header['DATE_END']
st = st.replace('T', ' ').split('.')[0]
et = et.replace('T', ' ').split('.')[0]

In [5]:
spec = data[0].data
spec.shape

(2, 1, 731, 53385)

In [6]:
spec = spec[:, 0, :, :]
spec.shape

(2, 731, 53385)

In [7]:
spec_pol = spec[0]
spec_pol.shape

(731, 53385)

In [8]:
freq = data['SFREQ'].data['SFREQ']
freq.shape

(731,)

In [9]:
ut = data['UT'].data
mjd  = ut['mjd']
ms   = ut['time']
time_mjd = mjd + ms / 86400000.0

In [10]:
time_mjd

array([61058.67361441, 61058.67362036, 61058.67362631, ...,
       61058.99651073, 61058.99651668, 61058.99652262])

In [11]:
from astropy.time import Time

t = Time(time_mjd, format='mjd')
time_dt = t.to_datetime()
time_dt

array([datetime.datetime(2026, 1, 18, 16, 10, 0, 285000),
       datetime.datetime(2026, 1, 18, 16, 10, 0, 799000),
       datetime.datetime(2026, 1, 18, 16, 10, 1, 313000), ...,
       datetime.datetime(2026, 1, 18, 23, 54, 58, 527000),
       datetime.datetime(2026, 1, 18, 23, 54, 59, 41000),
       datetime.datetime(2026, 1, 18, 23, 54, 59, 554000)], dtype=object)

In [ ]:
fig = plt.figure(figsize=[12,5])
ax  = fig.add_subplot(111)
dyspec = ax.pcolormesh(
    time_dt,
    freq*1e3,
    # np.log10(spec_pol),
    spec_pol,
    norm=LogNorm(vmin=np.nanpercentile(spec_pol, 5),
                 vmax=np.nanpercentile(spec_pol, 99)),
    cmap='RdYlBu_r'
)
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(f'OVRO-LWA Dynamic Spectrum: {st} - {et}')
plt.colorbar(dyspec, pad=0.01, label='Intensity (SFU)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
fig.tight_layout()
plt.show()

In [13]:
from datetime import datetime
from typing import Union, List, Tuple

def find_nearest_datetime(times: Union[List, np.ndarray], target: Union[str, datetime]) -> Tuple[int, datetime]:
    """
    Find the index and value of the datetime nearest to target.

    Parameters
    ----------
    times : list or np.ndarray
        List or array of datetime objects or ISO-format strings.
    target : str or datetime
        Target datetime (string in ISO format or datetime object).

    Returns
    -------
    nearest_idx : int
        Index of the nearest datetime in the array.
    nearest_time : datetime
        The nearest datetime object from the array.
    """
    # Convert target to datetime if string
    if isinstance(target, str):
        target_dt = datetime.fromisoformat(target)
    else:
        target_dt = target

    # Convert times to datetime objects if they are strings
    if isinstance(times[0], str):
        times_dt = [datetime.fromisoformat(t) for t in times]
    else:
        times_dt = times

    # Compute absolute differences in seconds
    differences = [abs((t - target_dt).total_seconds()) for t in times_dt]

    nearest_idx = differences.index(min(differences))
    return nearest_idx, times_dt[nearest_idx]

In [14]:
st_target = '2026-01-18T17:30:00'
et_target = '2026-01-18T19:30:00'

st_idx, st_nearest = find_nearest_datetime(time_dt, st_target)
et_idx, et_nearest = find_nearest_datetime(time_dt, et_target)

print(st_idx, st_nearest)
print(et_idx, et_nearest)

9228 2026-01-18 17:29:59.896000
23013 2026-01-18 19:30:00.210000


In [15]:
Time('2026-01-18 17:40:00', scale='utc', format='iso')

<Time object: scale='utc' format='iso' value=2026-01-18 17:40:00.000>

In [ ]:
# slice the spectrum
sub_spec = spec_pol[:, st_idx:et_idx]
sub_time = time_dt[st_idx:et_idx]

fig = plt.figure(figsize=[12,5])
ax  = fig.add_subplot(111)
dyspec = ax.pcolormesh(
    sub_time,
    freq*1e3,
    sub_spec,
    norm=LogNorm(vmin=np.nanpercentile(sub_spec, 5),
                 vmax=np.nanpercentile(sub_spec, 99.9)),
    cmap='RdYlBu_r'
)
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.set_title(f'OVRO-LWA Dynamic Spectrum: {sub_time[0]} - {sub_time[-1]}')
plt.colorbar(dyspec, pad=0.01, label='Intensity (SFU)')
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
# ax.set_xlim(left=sub_time[0], right=sub_time[-1])
ax.set_xlim(left=Time('2026-01-18 17:42:00', scale='utc', format='iso').to_datetime(),
            right=Time('2026-01-18 18:15:00', scale='utc', format='iso').to_datetime()
           )
fig.tight_layout()
plt.show()